# Phase 0 — validate the training loop

Stock PPO from `brax.training` on the stock `ant` env, MJX backend (ADR 0002).
No custom code beyond glue — the point is a **known-good loop**: training curve
rises, checkpoint round-trips, rollout renders. Every later debugging session
starts from here ([SPEC](../SPEC.md), Phase 0).

Colab: [open this notebook](https://colab.research.google.com/github/jamesponwith/brax-tennis-rl/blob/main/notebooks/phase0_ant.ipynb) —
Runtime → T4 GPU is plenty. Locally it runs in smoke mode via `PHASE0_SMOKE=1`
(tiny step count, CPU) to prove mechanics, not to learn a gait.

In [ ]:
try:
    import brax  # noqa: F401
except ImportError:  # Colab: match the repo's pinned version (ADR 0002)
    %pip install -q brax==0.14.2

import os

SMOKE = os.environ.get("PHASE0_SMOKE", "") == "1"
print(f"smoke mode: {SMOKE}")

In [ ]:
# hyperparams: brax's published ant-PPO settings; smoke mode shrinks
# everything to prove the mechanics on CPU in ~a minute
TRAIN = {
    "num_timesteps": 25_000_000,
    "num_evals": 20,
    "reward_scaling": 10,
    "episode_length": 1000,
    "normalize_observations": True,
    "action_repeat": 1,
    "unroll_length": 5,
    "num_minibatches": 32,
    "num_updates_per_batch": 4,
    "discounting": 0.97,
    "learning_rate": 3e-4,
    "entropy_cost": 1e-2,
    "num_envs": 4096,
    "batch_size": 2048,
    "seed": 1,
}
if SMOKE:
    TRAIN.update(
        num_timesteps=32_768,
        num_evals=2,
        episode_length=128,
        num_envs=64,
        batch_size=64,
        num_minibatches=4,
    )

In [ ]:
from brax import envs
from brax.training.agents.ppo import train as ppo

env = envs.get_environment("ant", backend="mjx")

xdata, ydata = [], []


def progress(num_steps, metrics):
    xdata.append(num_steps)
    ydata.append(float(metrics["eval/episode_reward"]))
    print(f"{num_steps:>12,} steps  reward {ydata[-1]:8.1f}")


make_inference_fn, params, _ = ppo.train(environment=env, progress_fn=progress, **TRAIN)

In [ ]:
import matplotlib.pyplot as plt

plt.plot(xdata, ydata, marker="o")
plt.xlabel("env steps")
plt.ylabel("eval episode reward")
plt.title("ant PPO — training curve")
plt.savefig("phase0_curve.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# checkpoint round-trip: save, reload, use the reloaded params from here on
from brax.io import model

model.save_params("ant_params", params)
params = model.load_params("ant_params")
print("checkpoint round-trip ok")

In [ ]:
import jax
from brax.io import html

inference_fn = make_inference_fn(params)
jit_reset = jax.jit(env.reset)
jit_step = jax.jit(env.step)
jit_infer = jax.jit(inference_fn)

rng = jax.random.PRNGKey(0)
state = jit_reset(rng=rng)
rollout = []
for _ in range(100 if SMOKE else 500):
    rollout.append(state.pipeline_state)
    act_rng, rng = jax.random.split(rng)
    act, _ = jit_infer(state.obs, act_rng)
    state = jit_step(state, act)

page = html.render(env.sys.tree_replace({"opt.timestep": env.dt}), rollout)
with open("phase0_rollout.html", "w") as f:
    f.write(page)
print(f"rendered {len(rollout)} frames -> phase0_rollout.html")

## Done when

- the curve rises (full run: ant walks by ~10M steps, reward in the thousands)
- `checkpoint round-trip ok` printed — save/load is how Phase 1 will resume tuning
- `phase0_rollout.html` opens and plays (download it from Colab's file pane;
  screen-record → gif for the writeup, per SPEC)

Next: bead `brax-tennis-rl-5fr` — the interception env (`envs/tennis.py`).